# Taller: monta tú el modelo

Segunda parte del seminario. Aquí no hay teoría nueva: repetimos el recorrido de las slides sobre otro conjunto de datos.

> **Pendiente de decidir el conjunto de datos del taller.** Mientras tanto, las celdas de carga están marcadas con `TODO` y el notebook se apoya en `data/radon.csv` para que todo ejecute de principio a fin.

El recorrido es el mismo de siempre:

1. Mirar los datos.
2. Un modelo con todo junto (*complete pooling*).
3. Un modelo por grupo, cada uno por su cuenta (*unpooled*).
4. Un modelo jerárquico, donde los grupos se prestan información.
5. Responder una pregunta concreta con la posteriori.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az

from src.estilo import ACENTO, aplicar_estilo

aplicar_estilo()
SEMILLA = 42

## 1. Los datos

**TODO**: sustituir por el conjunto de datos del taller.

Lo que necesita ese conjunto para que el taller funcione:

- una variable respuesta continua (o transformable a continua),
- al menos un predictor,
- una variable de grupo con **muchos grupos y muy distinto número de observaciones por grupo**. Ese desequilibrio es lo que hace interesante el modelo jerárquico.

In [ ]:
# TODO: cambiar por el fichero del taller
df = pd.read_csv(RAIZ / "data" / "radon.csv")

RESPUESTA = "log_radon"
PREDICTOR = "floor"
GRUPO = "county"
GRUPO_CODIGO = "county_code"

df[[RESPUESTA, PREDICTOR, GRUPO, GRUPO_CODIGO]].head()

### Ejercicio 1

Antes de tocar ningún modelo:

- ¿Cómo se distribuye la respuesta? ¿Hace falta transformarla?
- ¿Cuántos grupos hay y cuántas observaciones tiene cada uno?
- ¿Qué grupo tiene menos datos? Guárdalo: es el que más te va a enseñar al final.

In [ ]:
# Tu código aquí


## 2. Modelo agrupado (complete pooling)

$$y_i \sim \mathcal{N}(\alpha + \beta x_i, \; \sigma^2)$$

### Ejercicio 2

Antes de escribir las prioris, mira la escala de tu respuesta: media, desviación típica, mínimo y máximo. Elige las sigmas a partir de ahí, no a ojo.

In [ ]:
y = df[RESPUESTA].values
x = df[PREDICTOR].values
grupo_idx = df[GRUPO_CODIGO].values
n_grupos = df[GRUPO_CODIGO].nunique()

with pm.Model() as modelo_agrupado:
    alpha = pm.Normal("alpha", mu=0, sigma=2)  # TODO: justificar la sigma
    beta = pm.Normal("beta", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    pm.Normal("y_obs", mu=alpha + beta * x, sigma=sigma, observed=y)

    idata_agrupado = pm.sample(draws=1000, tune=1000, random_seed=SEMILLA)

### Ejercicio 3: comprobación predictiva previa

Simula desde las prioris **sin dejar que el modelo vea los datos** y compara el rango simulado con el observado. Si el modelo da por normales valores imposibles en tu dominio, las prioris están mal.

Pista: `pm.sample_prior_predictive()`.

In [ ]:
# Tu código aquí


## 3. Modelo por grupo (unpooled)

$$y_i \sim \mathcal{N}(\alpha_{g[i]} + \beta x_i, \; \sigma^2), \qquad \alpha_g \sim \mathcal{N}(0, 2^2)$$

### Ejercicio 4

Ajústalo y dibuja el intervalo de credibilidad del intercepto de cada grupo, ordenados por número de observaciones. ¿Qué les pasa a los grupos con pocos datos?

In [ ]:
# Tu código aquí


## 4. Modelo jerárquico (partial pooling)

$$
\begin{aligned}
y_i &\sim \mathcal{N}(\alpha_{g[i]} + \beta x_i, \; \sigma^2) \\
\alpha_g &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\
\mu_\alpha &\sim \mathcal{N}(0, 2^2), \qquad \sigma_\alpha \sim \text{Exp}(1)
\end{aligned}
$$

### Ejercicio 5

Respecto al anterior solo cambia una línea. Escríbela.

In [ ]:
with pm.Model() as modelo_jerarquico:
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=2)
    sigma_alpha = pm.Exponential("sigma_alpha", 1)

    # TODO (ejercicio 5): la línea que cambia
    alpha = pm.Normal("alpha", mu=mu_alpha, sigma=sigma_alpha, shape=n_grupos)

    beta = pm.Normal("beta", mu=0, sigma=1)
    sigma = pm.Exponential("sigma", 1)

    pm.Normal("y_obs", mu=alpha[grupo_idx] + beta * x, sigma=sigma, observed=y)

    idata_jerarquico = pm.sample(
        draws=1000, tune=1000, target_accept=0.95, random_seed=SEMILLA
    )

### Ejercicio 6: shrinkage

Dibuja el intercepto unpooled y el jerárquico de cada grupo frente a su número de observaciones, unidos por una línea. Después calcula $(\sigma/\sigma_\alpha)^2$ e interprétalo: ¿a cuántas observaciones propias equivale lo que el modelo aprende del resto de grupos?

In [ ]:
# Tu código aquí


### Ejercicio 7: diagnóstico

- `az.summary()`: mira `r_hat`, `ess_bulk` y `ess_tail`.
- Cuenta las divergencias: `idata_jerarquico.sample_stats["diverging"].sum()`.
- `az.plot_ppc()`: ¿reproduce el modelo los datos que ha visto? ¿Dónde falla?

Recuerda la distinción: `r_hat` y las divergencias dicen si el **muestreador** ha funcionado. La predictiva posterior dice si el **modelo** describe los datos. No son la misma pregunta.

In [ ]:
# Tu código aquí


## 5. La pregunta que le importa a alguien

### Ejercicio 8

Elige un grupo con muchos datos y otro con muy pocos. Para cada uno, calcula desde la posteriori:

- el intervalo del **nivel medio del grupo**,
- el intervalo de una **observación concreta** de ese grupo (que además arrastra $\sigma$),
- la probabilidad de superar un umbral que te importe.

Y explica en una frase, sin jerga, la diferencia entre los dos intervalos.

In [ ]:
# Tu código aquí


## Para llevarte a casa

- ¿Qué decisión tomarías con estos resultados y qué te haría falta para tomarla mejor?
- ¿Qué priori te ha costado más justificar? Esa es la parte del modelo que hay que defender.